# CUHK-X Small Model Track — Step 3: fuse, pack, submit

Three jobs:
1. Fit modality fusion weights **on out-of-fold predictions**, never on the
   leaderboard. Tuning weights against the public LB is how a team ends up
   with a great public score and a failed Selection Stage.
2. Pack every ensemble member into **one fp16 checkpoint under 100 MB**, which
   is what the rules require.
3. Write `submission.csv` in exactly the expected format.

**Setup:** attach the Step 2 notebook's output (the `.pt` / `.npz` files) and
the `compact` dataset. GPU only needed if you re-run inference here.

In [ ]:
import os, glob, json, itertools
import numpy as np, pandas as pd, torch

WORK = "/kaggle/working"
ART  = None                      # directory holding *_oof.npz / *_test.npz / *.pt
for c in glob.glob("/kaggle/input/*"):
    if glob.glob(f"{c}/*_oof.npz"):
        ART = c; break
ART = ART or WORK
DATA = next((p for p in glob.glob("/kaggle/input/*/compact") if os.path.isdir(p)),
            "/kaggle/input/cuhkx-compact/compact")
print("artifacts:", ART, "\ndata     :", DATA)
print(sorted(os.path.basename(p) for p in glob.glob(f"{ART}/*.np*"))[:20])

## Load OOF predictions

Each `{mod}_f{k}_oof.npz` covers a disjoint set of users, so concatenating the
folds of one modality gives a full out-of-fold prediction over all training
subjects — an unbiased estimate of cross-subject accuracy.

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

oof = {}      # modality -> DataFrame(clip_id, user, y, p0..p39)
for f in sorted(glob.glob(f"{ART}/*_oof.npz")):
    base = os.path.basename(f)[:-8]           # "Thermal_f0"
    mod = base.rsplit("_f", 1)[0]
    d = np.load(f, allow_pickle=True)
    p = softmax(d["logits"].astype(np.float64))
    df = pd.DataFrame(p, columns=[f"p{i}" for i in range(p.shape[1])])
    df.insert(0, "y", d["y"]); df.insert(0, "user", d["user"].astype(str))
    df.insert(0, "clip_id", d["clip_id"].astype(str))
    oof.setdefault(mod, []).append(df)

for m in list(oof):
    oof[m] = pd.concat(oof[m], ignore_index=True)
    P = oof[m][[f"p{i}" for i in range(40)]].values
    acc = (P.argmax(1) == oof[m].y.values).mean()
    print(f"{m:<14} {len(oof[m]):>6} OOF clips  "
          f"{oof[m].user.nunique():>2} users  acc {acc:.4f}")

if not oof:
    raise SystemExit("No *_oof.npz found — run Step 2 first.")

## Fit fusion weights on OOF

Grid search over simplex weights. With few modalities this is exact and far
more robust than learning a stacker on so little data.

In [ ]:
mods = sorted(oof)
common = set(oof[mods[0]].clip_id)
for m in mods[1:]:
    common &= set(oof[m].clip_id)
common = sorted(common)
print(f"clips shared by all {len(mods)} modalities: {len(common)}")

cols = [f"p{i}" for i in range(40)]
stack, y = [], None
for m in mods:
    d = oof[m].set_index("clip_id").loc[common]
    stack.append(d[cols].values)
    y = d.y.values
stack = np.stack(stack)                       # [M, N, 40]

best_w, best_acc = None, -1
grid = np.arange(0, 1.01, 0.05)
for w in itertools.product(grid, repeat=len(mods)):
    s = sum(w)
    if s <= 0: continue
    w = np.array(w) / s
    acc = ((w[:, None, None] * stack).sum(0).argmax(1) == y).mean()
    if acc > best_acc:
        best_acc, best_w = acc, w

for m in mods:
    d = oof[m].set_index("clip_id").loc[common]
    print(f"  {m:<14} solo {(d[cols].values.argmax(1)==y).mean():.4f}")
print(f"\nbest fusion weights: {dict(zip(mods, best_w.round(3)))}")
print(f"FUSED OOF cross-subject accuracy: {best_acc:.4f}")

### Where is it failing?

Per-class recall points at what to fix next. Confusions here are almost always
between the semantically adjacent classes (*Check the time* / *Use a mobile
phone* / *Make a phone call*), which is a spatial-detail problem, not a
temporal one.

In [ ]:
fused = (best_w[:, None, None] * stack).sum(0)
pred = fused.argmax(1)
cm_names = pd.read_csv(f"{DATA}/class_mapping.csv") \
    if os.path.exists(f"{DATA}/class_mapping.csv") else None
name = dict(zip(cm_names.action_id, cm_names.action_name)) if cm_names is not None else {}

rows = []
for c in range(40):
    m_ = y == c
    if m_.sum() == 0: continue
    wrong = pred[m_][pred[m_] != c]
    top = pd.Series(wrong).value_counts().head(1)
    rows.append(dict(cls=c, name=name.get(c, c), n=int(m_.sum()),
                     recall=float((pred[m_] == c).mean()),
                     top_confusion=name.get(int(top.index[0]), int(top.index[0]))
                     if len(top) else "-"))
worst = pd.DataFrame(rows).sort_values("recall").head(12)
print(worst.to_string(index=False))

## Build the submission

In [ ]:
test_p, test_ids = {}, None
for f in sorted(glob.glob(f"{ART}/*_test.npz")):
    base = os.path.basename(f)[:-9]
    mod = base.rsplit("_f", 1)[0]
    d = np.load(f, allow_pickle=True)
    ids = d["clip_id"].astype(str)
    df = pd.DataFrame(d["probs"], index=ids)
    test_p.setdefault(mod, []).append(df)

agg = {}
for m, lst in test_p.items():
    # average the folds of a modality, then align every modality to one index
    agg[m] = sum(x.reindex(lst[0].index) for x in lst) / len(lst)
    print(f"{m:<14} test clips {len(agg[m])}")

all_ids = sorted(set().union(*[set(v.index) for v in agg.values()]))
num, den = np.zeros((len(all_ids), 40)), np.zeros((len(all_ids), 1))
for w, m in zip(best_w, mods):
    if m not in agg: continue
    v = agg[m].reindex(all_ids)
    mask = ~v.isna().all(axis=1).values          # clips missing this modality
    num[mask] += w * np.nan_to_num(v.values[mask])
    den[mask] += w                                # renormalise over what exists
final = num / np.maximum(den, 1e-9)
pred = final.argmax(1)

sub = pd.DataFrame({"path": [f"small_model_track_test/{i}/" for i in all_ids],
                    "prediction": pred.astype(int)})

# Conform to the official template: same rows, same order, no gaps.
tmpl = pd.read_csv(f"{DATA}/test.csv")
sub = tmpl[["path"]].merge(sub, on="path", how="left")
missing = sub.prediction.isna().sum()
if missing:
    print(f"!! {missing} clips had no prediction — filling with the modal class")
    sub["prediction"] = sub.prediction.fillna(pd.Series(pred).mode()[0])
sub["prediction"] = sub.prediction.astype(int)

sub.to_csv(f"{WORK}/submission.csv", index=False)
print(f"\nwrote {len(sub)} rows -> submission.csv")
print(sub.head())
print(f"\nprediction spread: {sub.prediction.nunique()}/40 classes used")
print(sub.prediction.value_counts().head(5))

## Pack the competition checkpoint

Rule: *every* weight loaded at inference goes into one file under 100 MB.

In [ ]:
%%writefile cuhkx.py
"""
CUHK-X Small Model Track — shared training/inference library.

Design notes (why it is built this way):

* The competition is scored cross-subject: test users never appear in training.
  So validation MUST be grouped by user, otherwise CV is meaningless and you
  tune yourself off a cliff. Everything here uses GroupKFold on `user`.

* Backbone is ImageNet-pretrained ResNet18 with Temporal Shift Modules inserted
  into the residual branches. TSM gives 3D-conv-like temporal modelling at 2D
  cost and adds *zero* parameters, which matters under the 100 MB budget.
  Host confirmed ImageNet-pretrained small CNNs are allowed.

* Per-clip intensity normalisation is applied to thermal/IR. Absolute pixel
  level encodes body temperature and ambient conditions — i.e. subject and
  session identity — which is exactly the nuisance variable we must discard to
  generalise across people.

* Weights are exported fp16; a full modality x fold ensemble packs into one
  checkpoint well under 100 MB.
"""
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass, field, asdict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

cv2.setNumThreads(0)

NUM_CLASSES = 40


# ------------------------------------------------------------------- config

@dataclass
class Cfg:
    data_root: str = "/kaggle/input/cuhkx-compact"
    modality: str = "Thermal"
    frames: int = 16          # frames fed to the model
    size: int = 144           # train crop; stored frames are larger
    arch: str = "resnet18"
    n_folds: int = 5
    fold: int = 0
    epochs: int = 20
    batch_size: int = 16
    lr: float = 3e-4
    backbone_lr_mult: float = 0.3   # pretrained trunk moves slower than the head
    weight_decay: float = 0.05
    label_smoothing: float = 0.1
    mixup_alpha: float = 0.2
    mixup_prob: float = 0.5
    shift_div: int = 8        # TSM: fraction of channels shifted
    dropout: float = 0.3
    ema_decay: float = 0.999
    warmup_frac: float = 0.1
    grad_clip: float = 5.0
    amp: bool = True
    num_workers: int = 2
    seed: int = 42
    per_clip_norm: bool = True
    out_dir: str = "/kaggle/working"
    extra: dict = field(default_factory=dict)

    def to_json(self):
        return json.dumps(asdict(self), indent=2, default=str)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------------- dataset

class BlobReader:
    """Random access into the packed JPEG blobs written by 04_preprocess.py."""

    def __init__(self, mod_dir):
        self.mod_dir = mod_dir
        self._files = {}

    def read(self, blob_id, offset, length):
        f = self._files.get(blob_id)
        if f is None:
            f = open(os.path.join(self.mod_dir, f"blob_{blob_id:03d}.bin"), "rb")
            self._files[blob_id] = f
        f.seek(offset)
        return f.read(length)

    def __getstate__(self):
        # File handles must not cross the fork into dataloader workers.
        return {"mod_dir": self.mod_dir, "_files": {}}

    def __setstate__(self, s):
        self.__dict__.update(s)
        self._files = {}


def load_index(data_root, modality):
    mod_dir = os.path.join(data_root, modality)
    idx = pd.read_parquet(os.path.join(mod_dir, "index.parquet"))
    return mod_dir, idx


class ClipDataset(Dataset):
    """
    Yields (clip_tensor[T,C,H,W], label).

    Temporal sampling: the preprocessor already stored T_stored uniformly-spaced
    frames. At train time we jitter *within* that grid so the model sees
    different phases of the action across epochs.
    """

    def __init__(self, df, mod_dir, cfg: Cfg, train: bool):
        self.df = df.reset_index(drop=True)
        self.reader = BlobReader(mod_dir)
        self.cfg = cfg
        self.train = train

    def __len__(self):
        return len(self.df)

    # -- frame decode -------------------------------------------------------
    def _decode(self, row, take):
        offs, lens = row["offsets"], row["lengths"]
        out = []
        for i in take:
            buf = self.reader.read(int(row["blob"]), int(offs[i]), int(lens[i]))
            a = np.frombuffer(buf, np.uint8)
            img = cv2.imdecode(a, cv2.IMREAD_COLOR)
            if img is None:
                img = np.zeros((self.cfg.size, self.cfg.size, 3), np.uint8)
            out.append(img)
        return out

    def _pick(self, n_stored):
        T = self.cfg.frames
        if n_stored <= T:
            base = list(range(n_stored)) + [n_stored - 1] * (T - n_stored)
            return base
        if self.train:
            # segment-based random sampling (TSN style): one random frame per segment
            edges = np.linspace(0, n_stored, T + 1)
            return [int(np.random.randint(edges[i], max(edges[i] + 1, edges[i + 1])))
                    for i in range(T)]
        edges = np.linspace(0, n_stored, T + 1)
        return [int((edges[i] + edges[i + 1]) / 2) for i in range(T)]

    # -- augmentation -------------------------------------------------------
    def _augment(self, imgs):
        cfg = self.cfg
        H, W = imgs[0].shape[:2]
        if self.train:
            scale = np.random.uniform(0.65, 1.0)
            ar = np.random.uniform(0.85, 1.18)
            ch = int(min(H, H * scale * ar))
            cw = int(min(W, W * scale / ar))
            y0 = np.random.randint(0, H - ch + 1)
            x0 = np.random.randint(0, W - cw + 1)
            flip = np.random.rand() < 0.5
            # brightness/contrast jitter — models sensor gain drift, not identity
            alpha = np.random.uniform(0.85, 1.15)
            beta = np.random.uniform(-12, 12)
        else:
            side = int(min(H, W) * 0.90)
            y0 = (H - side) // 2
            x0 = (W - side) // 2
            ch = cw = side
            flip, alpha, beta = False, 1.0, 0.0

        out = []
        for im in imgs:
            im = im[y0:y0 + ch, x0:x0 + cw]
            im = cv2.resize(im, (cfg.size, cfg.size), interpolation=cv2.INTER_LINEAR)
            if flip:
                im = im[:, ::-1]
            if self.train and (alpha != 1.0 or beta != 0.0):
                im = np.clip(im.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
            out.append(im)
        return out

    def __getitem__(self, i):
        row = self.df.iloc[i]
        take = self._pick(int(row["t"]))
        imgs = self._decode(row, take)
        imgs = self._augment(imgs)

        x = np.stack(imgs).astype(np.float32) / 255.0     # [T,H,W,3]
        x = torch.from_numpy(x).permute(0, 3, 1, 2)        # [T,3,H,W]

        if self.cfg.per_clip_norm:
            # Standardise each clip independently: removes the absolute thermal /
            # IR offset that identifies the subject and the session.
            m, s = x.mean(), x.std().clamp_min(1e-4)
            x = (x - m) / s
        else:
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
            x = (x - mean) / std

        if self.train and np.random.rand() < 0.25:
            x = random_erase(x)

        return x, int(row["action_id"])


def random_erase(x, max_frac=0.20):
    T, C, H, W = x.shape
    h = int(H * np.random.uniform(0.06, max_frac))
    w = int(W * np.random.uniform(0.06, max_frac))
    y = np.random.randint(0, H - h + 1)
    xx = np.random.randint(0, W - w + 1)
    x[:, :, y:y + h, xx:xx + w] = torch.randn(T, C, h, w) * 0.1
    return x


# --------------------------------------------------------------------- model

class TemporalShift(nn.Module):
    """
    Shift a fraction of channels forward/backward along time before `block`.

    Cost: a memory copy. Params added: zero. This is what buys temporal
    reasoning without paying for 3D convolutions.
    """

    def __init__(self, block, n_segment, shift_div=8):
        super().__init__()
        self.block = block
        self.n_segment = n_segment
        self.shift_div = shift_div

    def forward(self, x):
        nt, c, h, w = x.size()
        t = self.n_segment
        n = nt // t
        x = x.view(n, t, c, h, w)
        fold = max(1, c // self.shift_div)
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]              # shift left  (future)
        out[:, 1:, fold:2 * fold] = x[:, :-1, fold:2 * fold]  # shift right (past)
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]         # keep the rest
        return self.block(out.view(nt, c, h, w))


def make_tsm_resnet(arch="resnet18", n_segment=16, shift_div=8, pretrained=True):
    import torchvision
    fn = getattr(torchvision.models, arch)
    try:
        net = fn(weights="IMAGENET1K_V1" if pretrained else None)
    except TypeError:
        net = fn(pretrained=pretrained)

    # Wrap the first conv of every BasicBlock so the shift happens on the
    # residual branch only — identity path stays clean (as in the TSM paper).
    for layer in [net.layer1, net.layer2, net.layer3, net.layer4]:
        for blk in layer:
            blk.conv1 = TemporalShift(blk.conv1, n_segment, shift_div)
    feat_dim = net.fc.in_features
    net.fc = nn.Identity()
    return net, feat_dim


class VideoNet(nn.Module):
    """TSM backbone + attention-weighted temporal pooling + linear classifier."""

    def __init__(self, cfg: Cfg, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        self.cfg = cfg
        self.backbone, d = make_tsm_resnet(
            cfg.arch, cfg.frames, cfg.shift_div, pretrained)
        self.attn = nn.Sequential(nn.Linear(d, d // 4), nn.Tanh(), nn.Linear(d // 4, 1))
        self.drop = nn.Dropout(cfg.dropout)
        self.fc = nn.Linear(d, num_classes)
        nn.init.trunc_normal_(self.fc.weight, std=0.01)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):                 # x: [B,T,3,H,W]
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))         # [B*T, d]
        f = f.view(B, T, -1)
        a = self.attn(f).softmax(dim=1)            # [B,T,1]
        pooled = (f * a).sum(1)                    # [B,d]
        return self.fc(self.drop(pooled))


# ------------------------------------------------------- skeleton (ST-GCN)

# Human3.6M 17-joint bone list; parent[j] is the joint j hangs off.
H36M_EDGES = [(0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6),
              (0, 7), (7, 8), (8, 9), (9, 10),
              (8, 11), (11, 12), (12, 13), (8, 14), (14, 15), (15, 16)]
N_JOINTS = 17
PARENT = np.zeros(N_JOINTS, dtype=np.int64)
for _p, _c in H36M_EDGES:
    PARENT[_c] = _p


def build_adjacency():
    """
    Three-partition spatial graph (ST-GCN): self, centripetal (towards root),
    centrifugal (away). Splitting by distance-to-root lets the convolution treat
    "limb moving inward" and "limb moving outward" differently, which matters
    for reach/retract actions like *Take medicine* vs *Put on clothes*.
    """
    A = np.zeros((N_JOINTS, N_JOINTS), np.float32)
    for i, j in H36M_EDGES:
        A[i, j] = A[j, i] = 1.0

    # hop distance from the root joint
    dist = np.full(N_JOINTS, 1e9)
    dist[0] = 0
    for _ in range(N_JOINTS):
        for i, j in H36M_EDGES:
            dist[j] = min(dist[j], dist[i] + 1)
            dist[i] = min(dist[i], dist[j] + 1)

    parts = np.zeros((3, N_JOINTS, N_JOINTS), np.float32)
    parts[0] = np.eye(N_JOINTS, dtype=np.float32)
    for i in range(N_JOINTS):
        for j in range(N_JOINTS):
            if A[i, j] == 0:
                continue
            if dist[j] < dist[i]:
                parts[1, i, j] = 1.0      # neighbour closer to root
            else:
                parts[2, i, j] = 1.0      # neighbour further from root

    # symmetric normalisation, per partition
    for k in range(3):
        d = parts[k].sum(1, keepdims=True)
        parts[k] = parts[k] / np.maximum(d, 1e-6)
    return torch.from_numpy(parts)


def pose_features(x):
    """
    [B,T,V,3] -> [B,9,T,V]: joint position, bone vector, and velocity.

    Bones encode limb orientation independently of where the joint sits, and
    velocity supplies the short-term dynamics that separate the exercise classes
    (jog / squat / jumping jack) from each other.
    """
    B, T, V, C = x.shape
    parent = torch.as_tensor(PARENT, device=x.device)
    bone = x - x[:, :, parent, :]
    vel = torch.zeros_like(x)
    vel[:, 1:] = x[:, 1:] - x[:, :-1]
    f = torch.cat([x, bone, vel], dim=-1)          # [B,T,V,9]
    return f.permute(0, 3, 1, 2).contiguous()      # [B,9,T,V]


class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.1, residual=True):
        super().__init__()
        self.register_buffer("A", A)
        K = A.size(0)
        self.gcn = nn.Conv2d(cin, cout * K, 1)
        self.K, self.cout = K, cout
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(cout), nn.Dropout(dropout),
        )
        # learnable edge weighting — lets the graph adapt beyond the skeleton
        self.edge = nn.Parameter(torch.ones(K, A.size(1), A.size(2)))
        if not residual:
            self.res = None
        elif cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)),
                                     nn.BatchNorm2d(cout))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = 0 if self.res is None else self.res(x)
        y = self.gcn(x)
        N, _, T, V = y.shape
        y = y.view(N, self.K, self.cout, T, V)
        y = torch.einsum("nkctv,kvw->nctw", y, self.A * self.edge)
        return self.relu(self.tcn(y) + res)


class SkeletonNet(nn.Module):
    """
    Compact ST-GCN over 3D pose. ~1 M parameters, so it costs almost nothing
    against the 100 MB budget while adding a modality that is present for 100%
    of clips and is far more subject-invariant than appearance.
    """

    def __init__(self, num_classes=NUM_CLASSES, width=64, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        w = width
        self.bn = nn.BatchNorm1d(9 * N_JOINTS)
        self.blocks = nn.ModuleList([
            STGCNBlock(9, w, A, residual=False),
            STGCNBlock(w, w, A),
            STGCNBlock(w, w, A),
            STGCNBlock(w, 2 * w, A, stride=2),
            STGCNBlock(2 * w, 2 * w, A),
            STGCNBlock(2 * w, 4 * w, A, stride=2),
            STGCNBlock(4 * w, 4 * w, A),
        ])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(4 * w, num_classes)

    def forward(self, x):                       # x: [B,T,V,3]
        f = pose_features(x)                    # [B,9,T,V]
        B, C, T, V = f.shape
        f = self.bn(f.permute(0, 1, 3, 2).reshape(B, C * V, T))
        f = f.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        for b in self.blocks:
            f = b(f)
        f = f.mean(dim=(2, 3))                  # global average over time+joints
        return self.fc(self.drop(f))


class SkeletonDataset(Dataset):
    """Yields ([T,17,3] pose, label) from the packed poses.npy."""

    def __init__(self, df, mod_dir, cfg: Cfg, train: bool):
        self.df = df.reset_index(drop=True)
        self.poses = np.load(os.path.join(mod_dir, "poses.npy"), mmap_mode="r")
        self.cfg = cfg
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        seq = np.asarray(self.poses[int(row["row"])], dtype=np.float32)  # [T,V,3]
        T = self.cfg.frames

        if seq.shape[0] != T:                       # resample along time
            src = np.linspace(0, seq.shape[0] - 1, T)
            if self.train:
                src = np.clip(src + np.random.uniform(-0.5, 0.5, T), 0,
                              seq.shape[0] - 1)
            seq = seq[np.round(src).astype(int)]

        if self.train:
            # Rotation about the vertical axis: the camera yaw relative to the
            # subject is arbitrary, so the label must be invariant to it.
            th = np.random.uniform(-0.35, 0.35)
            c, s = np.cos(th), np.sin(th)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float32)
            seq = seq @ R.T
            seq = seq * np.random.uniform(0.9, 1.1)
            seq = seq + np.random.normal(0, 0.01, seq.shape).astype(np.float32)
            if np.random.rand() < 0.5:              # mirror left/right
                seq[..., 0] *= -1
                swap = np.arange(N_JOINTS)
                for a, b in [(1, 4), (2, 5), (3, 6), (11, 14), (12, 15), (13, 16)]:
                    swap[a], swap[b] = b, a
                seq = seq[:, swap]

        return torch.from_numpy(np.ascontiguousarray(seq)), int(row["action_id"])


def make_model(cfg: Cfg, pretrained=True):
    """Dispatch on modality: pose gets the graph net, images get TSM."""
    if cfg.modality.lower().startswith("skel"):
        return SkeletonNet(dropout=cfg.dropout)
    return VideoNet(cfg, pretrained=pretrained)


def make_dataset(df, mod_dir, cfg: Cfg, train: bool):
    if cfg.modality.lower().startswith("skel"):
        return SkeletonDataset(df, mod_dir, cfg, train)
    return ClipDataset(df, mod_dir, cfg, train)


# ------------------------------------------------------------------ training

class EMA:
    """
    Exponential moving average of weights, with a warmup ramp on the decay.

    The shadow starts as a copy of the *untrained* weights, so a flat 0.999
    decay leaves it pinned near initialisation for the first ~1000 steps — on a
    short run you would evaluate and checkpoint an essentially untrained model.
    Ramping the decay as (1+t)/(10+t) makes the average track closely at first
    and tighten as training proceeds.
    """

    def __init__(self, model, decay):
        self.decay = decay
        self.step = 0
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)

    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v.to(sd[k].dtype))


def mixup(x, y, alpha):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam


def cosine_schedule(step, total, warmup):
    if step < warmup:
        return step / max(warmup, 1)
    p = (step - warmup) / max(total - warmup, 1)
    return 0.5 * (1 + math.cos(math.pi * p))


def build_folds(df, n_folds, seed=42):
    """
    GroupKFold on user. Deterministic, and balanced by user count rather than
    row count so each fold holds out a comparable number of *people*.
    """
    users = sorted(df["user"].astype(str).unique())
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(users))
    assign = {users[u]: i % n_folds for i, u in enumerate(order)}
    return df["user"].astype(str).map(assign).values


@torch.no_grad()
def predict_logits(model, loader, device, amp=True):
    model.eval()
    outs, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=amp and device.type == "cuda"):
            outs.append(model(x).float().cpu())
        ys.append(y)
    return torch.cat(outs), torch.cat(ys)


def pack_fp16(state_dicts, path, meta=None):
    """Bundle every ensemble member into ONE fp16 checkpoint (rule requirement)."""
    packed = {
        name: {k: v.half() if v.dtype.is_floating_point else v
               for k, v in sd.items()}
        for name, sd in state_dicts.items()
    }
    torch.save({"models": packed, "meta": meta or {}}, path)
    mb = os.path.getsize(path) / 1e6
    print(f"checkpoint: {path}  {mb:.1f} MB  ({len(packed)} models)")
    if mb > 100:
        print("!! OVER the 100 MB limit — drop members or shrink the backbone")
    return mb

In [ ]:
from cuhkx import pack_fp16
sds = {}
for p in sorted(glob.glob(f"{ART}/*.pt")):
    sds[os.path.basename(p)[:-3]] = torch.load(p, map_location="cpu")
if sds:
    mb = pack_fp16(sds, f"{WORK}/model.pth",
                   meta=dict(modalities=mods, weights=best_w.tolist(),
                             oof_acc=float(best_acc), frames=16, size=144,
                             arch="tsm_resnet18"))
    print(f"members: {list(sds)}")
else:
    print("no .pt files found to pack")

## Submit

`Save Version -> Save & Run All`, then from the notebook's Output tab click
**Submit to Competition**.

Track both numbers every time: the fused **OOF cross-subject accuracy** above
and the public LB. If the LB runs far ahead of OOF, you are fitting the test
set and will lose the seat at the Selection Stage — the pass criterion there
is a drop of no more than 10 points on fresh subjects.